# SeamlessM4T — son arabe → français, sur le jeu commun

**Lancer sur Colab ou Kaggle.** Depuis la racine du dépôt, avec la traduction de Paulo dans `donnees/marque/` :

```
zip -r NLP.zip code configs notebooks donnees/marque donnees/noms -x '*/__pycache__/*'
```

- Kaggle : `NLP.zip` en *Add Input*, GPU T4, Internet activé, secret `HF_TOKEN` (*Add-ons > Secrets*). La sortie `/kaggle/working/resultats.zip` s'ajoute en entrée de l'autre notebook.
- Colab : exécution GPU T4, secret `HF_TOKEN` (icône clé), envoyer `NLP.zip` (et `resultats.zip` de l'autre notebook) à l'invite. La dernière cellule télécharge `resultats.zip`.

In [3]:
# Colab ou Kaggle : installe, retrouve le projet (NLP.zip), lit le secret HF_TOKEN.
# Kaggle : NLP.zip en « Add Input », GPU T4 et Internet activés, secret HF_TOKEN dans Add-ons > Secrets.
# Colab : envoie NLP.zip à l'invite, secret HF_TOKEN dans le panneau « clé ».
# resultats.zip (sortie de l'autre notebook) est relu s'il est envoyé ou ajouté en entrée.
import os
import shutil
import sys
import zipfile
from pathlib import Path

SUR_COLAB = "google.colab" in sys.modules
SUR_KAGGLE = Path("/kaggle/working").is_dir()
if SUR_COLAB or SUR_KAGGLE:
    !pip install -q sentencepiece soundfile sacrebleu py3langid pyyaml unbabel-comet huggingface_hub
    travail = Path("/content") if SUR_COLAB else Path("/kaggle/working")
    entrees = Path("/content") if SUR_COLAB else Path("/kaggle/input")
    chercher = entrees.glob if SUR_COLAB else entrees.rglob  # pas de parcours de Google Drive

    def trouver_projet(dossier):
        return next((p.parents[2] for p in Path(dossier).rglob("code/seamless/benchmark_m4t.py")), None)

    # Le code est recopié à chaque lancement : un NLP.zip plus récent remplace l'ancien projet.
    # resultats/ n'est pas dans NLP.zip, donc les traductions déjà écrites restent.
    source = trouver_projet(entrees) if SUR_KAGGLE else None
    archives = sorted(chercher("NLP*.zip"), key=lambda p: p.stat().st_mtime)
    if source is not None:
        shutil.copytree(source, travail / "projet", dirs_exist_ok=True)  # /kaggle/input est en lecture seule
    elif archives:
        zipfile.ZipFile(archives[-1]).extractall(travail / "projet")
    elif trouver_projet(travail / "projet") is None:
        if SUR_COLAB:
            from google.colab import files
            files.upload()  # NLP.zip, et resultats.zip de l'autre notebook s'il existe
            archives = sorted(chercher("NLP*.zip"), key=lambda p: p.stat().st_mtime)
        if not archives:
            raise FileNotFoundError("NLP.zip introuvable : Kaggle → Add Input ; Colab → relance la cellule et envoie-le.")
        zipfile.ZipFile(archives[-1]).extractall(travail / "projet")
    projet = trouver_projet(travail / "projet")
    for archive in sorted(chercher("resultats*.zip")):
        if not (projet / "resultats").is_dir():
            zipfile.ZipFile(archive).extractall(projet)
            print("Résultats relus :", archive)

    if not os.environ.get("HF_TOKEN"):
        try:
            if SUR_COLAB:
                from google.colab import userdata
                os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
            else:
                from kaggle_secrets import UserSecretsClient
                os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as exc:
            print("Secret HF_TOKEN absent : MGB-2, COMET et COMETKiwi seront refusés.", type(exc).__name__)
    os.chdir(projet / "notebooks")
    print("Projet :", projet)

In [4]:
import json

MODE_ESSAI = os.environ.get("MODE_ESSAI", "0") == "1"

racine = Path.cwd()
while not (racine / "code" / "traduction" / "benchmark_traduction.py").exists() and racine.parent != racine:
    racine = racine.parent
if not (racine / "code" / "seamless" / "benchmark_m4t.py").exists():
    raise FileNotFoundError("Lance le notebook depuis le dépôt NLP (le dossier qui contient code/).")
for dossier in (racine / "code" / "traduction", racine / "code" / "seamless"):
    if str(dossier) not in sys.path:
        sys.path.insert(0, str(dossier))

import importlib

import benchmark_traduction as bt
import benchmark_m4t as m4t

# relance de la cellule = code à jour, sans redémarrer le noyau
bt = importlib.reload(bt)
m4t = importlib.reload(m4t)

config = bt.charger_config()
sortie = racine / config["donnees"]["dossier_sortie"]
if MODE_ESSAI:
    sortie = sortie / "essai"
sortie.mkdir(parents=True, exist_ok=True)
candidat = m4t.CANDIDAT_SEAMLESS
print("Sortie :", sortie)
print("Modèle :", candidat["modele_id"], "|", candidat["licence"])

Sortie : /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/resultats/traduction/5331FFF2
Modèle : facebook/seamless-m4t-v2-large | CC-BY-NC 4.0


In [5]:
segments, chemin_fr = bt.charger_jeu(config, sortie, racine)
episodes = sorted({s["episode"] for s in segments})
nb_ref = sum(1 for s in segments if s.get("fr_reference"))
print(f"{len(segments)} segments, {sum(s['duree_s'] for s in segments) / 60:.1f} min de son, émissions : {episodes}")
print(f"Référence française : {chemin_fr or 'aucune'} ({nb_ref}/{len(segments)} segments)")

217 segments, 24.9 min de son, émissions : ['5331FFF2-358E-445E-BA71-74B745387304']
Référence française : aucune (0/217 segments)


In [6]:
dossiers_audio = [racine / "donnees", racine / "donnees" / "audio"]
cache = racine / "donnees" / "externe" / "cache_mgb2"
MARGE_S = 0.5  # horodatages à la seconde : on élargit chaque tranche pour ne pas couper de mot

if MODE_ESSAI:
    import wave
    faux = sortie / "audio_factice"
    faux.mkdir(exist_ok=True)
    for episode in episodes:
        fin = max(s["fin"] for s in segments if s["episode"] == episode) + 1
        with wave.open(str(faux / f"{episode}.wav"), "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
            w.writeframes(b"\x00\x00" * int(fin * 16000))
    dossiers_audio = [faux]
    cache = sortie / "cache_factice"
else:
    # L'émission n'est pas dans le train : on la cherche dans dev puis test (1 Go chacun).
    # Licence QCRI à accepter sur https://huggingface.co/datasets/QCRI/mgb2, puis HF_TOKEN ou `hf auth login`.
    for nom_archive in ("dev/dev.tar.gz", "test/test.tar.gz"):
        if all(m4t.trouver_audio_episode(e, dossiers_audio + [cache / "emissions"]) for e in episodes):
            break
        archive = racine / "donnees" / "audio" / nom_archive
        if not archive.is_file():
            if SUR_COLAB:
                !pip install -q huggingface_hub
            from huggingface_hub import hf_hub_download
            print("Téléchargement de", nom_archive, "(1 Go)")
            hf_hub_download("QCRI/mgb2", nom_archive, repo_type="dataset", local_dir=racine / "donnees" / "audio")
        trouves, _ = m4t.extraire_episodes_archive(archive, episodes, cache / "emissions")
        print(f"{nom_archive} : {sorted(trouves) or 'pas cette émission'}")

extraits = m4t.preparer_extraits_xml(segments, dossiers_audio, cache, marge_s=MARGE_S)
print(f"{len(extraits)} extraits prêts dans {cache / 'extraits_xml'}")

from IPython.display import Audio, display
print(extraits[0]["id"], "|", extraits[0]["ar"])
display(Audio(extraits[0]["chemin_audio"]))

217 extraits prêts dans /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/donnees/externe/cache_mgb2/extraits_xml
5331FFF2-358E-445E-BA71-74B745387304_utt_1 | أهلا بكم، دعا الرئيس الفرنسي فرانسوا هولاند مواطنيه إلى الوحدة الوطنية التي اعتبرها سلاح فرنسا الوحيد في مواجهة التطرف،


In [7]:
class SeamlessFactice:
    def traduire(self, lot):
        return [("Le présentateur parle de la situation (seamless).", 0.01) for _ in lot]

    def fermer(self):
        pass


def fabrique(c):
    if MODE_ESSAI:
        return SeamlessFactice()
    print(f"Chargement de {c['modele_id']} (environ 9 Go la première fois)")
    return m4t.TraducteurSeamless(c["modele_id"], nb_faisceaux=5)


lignes = bt.traduire_candidat(candidat, extraits, sortie, fabrique)
traductions = {candidat["nom"]: lignes}
temps = sum(l["temps_s"] for l in lignes)
duree = sum(e["duree_s"] for e in extraits)
rtf = m4t.calculer_rtf(temps, duree)
print(f"{len(lignes)} traductions, {temps / len(lignes):.2f} s par segment, RTF {rtf}")
print("Fichier :", sortie / f"traductions_{candidat['nom']}.jsonl")

Chargement de facebook/seamless-m4t-v2-large (environ 9 Go la première fois)


/Users/ftayma24/miniconda3/envs/pnlp/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 18:47:39.170000 46975 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Fetching 2 files: 100%|██████████| 2/2 [11:21<00:00, 340.98s/it]
[transformers] Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.
Loading weights: 100%|██████████| 2232/2232 [00:00<00:00, 7489.51it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer SeamlessM4TTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is des

217 traductions, 5.63 s par segment, RTF 0.8169
Fichier : /Users/ftayma24/SKOOL/ING3/cours-nlp/NLP/resultats/traduction/5331FFF2/traductions_seamless-m4t-v2.jsonl


In [8]:
automatique = bt.controles_automatiques(segments, traductions)
cometkiwi = {} if MODE_ESSAI else bt.scores_cometkiwi(segments, traductions)
r = automatique[candidat["nom"]]
print(f"Inutilisables : {r['echec_dur_pct']} % | COMETKiwi : {cometkiwi.get(candidat['nom'], '-')} | "
      f"car./s médian : {r['caracteres_par_s_median']} | signaux : {r['signaux_pct']}")
if cometkiwi.get("_motif"):
    print("COMETKiwi non mesuré :", cometkiwi["_motif"])

autres = bt.charger_traductions_supplementaires(sortie, segments, [candidat["nom"]])
tous = {**autres, **traductions}
reference = bt.scores_avec_reference(segments, tous, avec_comet=not MODE_ESSAI)
if reference:
    print(f"\nContre la traduction de référence ({next(iter(reference.values()))['nb']} segments) :")
    for nom, sc in reference.items():
        print(f"  {nom:16s} chrF++ {sc['chrf_pp']}  BLEU {sc['bleu']}  COMET {sc['comet']}")

par_id = {nom: {l["id"]: l["fr"] for l in ls} for nom, ls in tous.items()}
for s in segments[:3]:
    print("\nAR  :", s["ar"])
    if s.get("fr_reference"):
        print(f"  {'référence':16s}", s["fr_reference"])
    for nom, fr in par_id.items():
        print(f"  {nom:16s}", fr.get(s["id"], ""))

if not MODE_ESSAI:
    ref = {candidat["nom"]: reference[candidat["nom"]]} if candidat["nom"] in reference else {}
    bt.enregistrer_dans_resultats(automatique, {"avec_reference": ref}, cometkiwi, [candidat], jeu=config["donnees"]["jeu"])
    from tableau_resultats import enregistrer_resultat
    enregistrer_resultat(systeme=f"TRAD:{candidat['nom']}", version_modele=candidat["modele_id"],
                         jeu=config["donnees"]["jeu"], mesure="RTF", valeur=rtf,
                         intervalle="temps_calcul/duree_son", configuration="configs/benchmark_traduction.yaml")
    print("\nLignes ajoutées à resultats/resultats.csv")

Inutilisables : 1.4 % | COMETKiwi : non mesuré | car./s médian : 17.6 | signaux : {'chiffre_manquant': 0.9, 'longueur_suspecte': 3.2, 'pas_francais': 1.4}
COMETKiwi non mesuré : No module named 'comet'

AR  : أهلا بكم، دعا الرئيس الفرنسي فرانسوا هولاند مواطنيه إلى الوحدة الوطنية التي اعتبرها سلاح فرنسا الوحيد في مواجهة التطرف،
  seamless-m4t-v2  Bonjour, le président français François Hollande a appelé ses citoyens à l'unité nationale qu'il considère comme l'arme de France dans la lutte contre l'extrémisme.

AR  : وكانت مصادر قضائية أكدت وقوع حوادث لإطلاق نار على عدة مساجد في فرنسا
  seamless-m4t-v2  Et des sources judiciaires ont confirmé l'occurrence d'accidents de tir sur plusieurs mosquées en France depuis

AR  : منذ وقوع الهجوم على مقر صحيفة شارلي إيبدو نهار الأربعاء. نتوقف مع هذا الخبر لنناقشه في محورين:
  seamless-m4t-v2  Depuis l'attaque sur le siège du journal Charlie Hebdo, le mercredi, nous nous arrêtons avec cette nouvelle pour la discuter dans deux axes.

Lignes ajoutées à

python(48990) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:

if SUR_COLAB or SUR_KAGGLE:
    archive = shutil.make_archive(str(travail / "resultats"), "zip", root_dir=racine, base_dir="resultats")
    print("Résultats :", archive)
    if SUR_COLAB:
        from google.colab import files
        files.download(archive)